In [47]:
from langchain_community.document_loaders import WebBaseLoader

In [48]:
loader=WebBaseLoader('https://docs.langchain.com/oss/python/integrations/document_loaders/index#document-loader-integrations')

In [49]:
data=loader.load()

In [50]:
data=data[0].page_content

In [51]:
#data chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
chunks_data=splitter.split_text(data)
chunks_data[0]

'Document loader integrations - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen'

In [52]:
#embidings
from langchain_ollama import OllamaEmbeddings
embeddings =OllamaEmbeddings(model="mxbai-embed-large")

In [53]:
vectors=embeddings.embed_documents(chunks_data)

In [54]:
vectors

[[-0.017548716,
  -0.009009317,
  0.041771095,
  0.0074691335,
  -0.005332203,
  0.00560591,
  -0.033832595,
  -0.029149653,
  -0.0005729703,
  0.056123238,
  -0.01573775,
  0.03650241,
  0.01157497,
  0.0080300495,
  -0.049108747,
  0.022698106,
  -0.013619768,
  -0.016193505,
  -0.08538707,
  0.020310422,
  -0.025661597,
  0.020914566,
  -0.06489756,
  0.012134881,
  -0.0019308556,
  0.056056026,
  -0.004232777,
  0.0144785335,
  0.09147126,
  0.051252313,
  -0.022807842,
  0.0018044054,
  0.009456592,
  -0.004906155,
  0.0014972878,
  -0.055936318,
  0.012090878,
  -0.021775791,
  0.017370827,
  -0.044748887,
  -0.016412115,
  0.014087451,
  0.004549056,
  -0.07934049,
  -0.06439022,
  0.044755902,
  0.030077875,
  0.038829066,
  -0.015783852,
  -0.02043218,
  -0.017224502,
  0.032000653,
  -0.015320717,
  -0.015430549,
  0.0077448306,
  -0.0073571317,
  -0.03098149,
  -0.017715754,
  -0.036840577,
  0.02984267,
  0.035661485,
  0.051588863,
  0.029746989,
  -0.023544364,
  0.007828

In [55]:
#store in vector store
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_texts(chunks_data,embedding=embeddings)

In [56]:
question="why we need document loaders"
answer=vector_store.similarity_search_with_score(question)
answer

[(Document(id='34d63fea-706e-4b73-a01e-ded94a061368', metadata={}, page_content='\u200bAll document loaders'),
  np.float32(0.19990623)),
 (Document(id='f9ddd0d9-7028-47f4-80e8-07322f57fc36', metadata={}, page_content='\u200bPDFs\nThe below document loaders allow you to load PDF documents.'),
  np.float32(0.3376888)),
 (Document(id='4a2cd3e4-d62a-4da2-bdca-f540303a4a6a', metadata={}, page_content='\u200bBy category\n\u200bWebpages\nThe below document loaders allow you to load webpages.'),
  np.float32(0.3664348)),
 (Document(id='fbe25f70-99b1-4793-8540-18545fed2733', metadata={}, page_content='The below document loaders allow you to load data from commonly used productivity tools.'),
  np.float32(0.38808465))]

In [57]:
vector_store.similarity_search_with_score(question)[0][1]

np.float32(0.19990623)

In [80]:
best_score=answer[0][1]
best_ans=answer[0][0].page_content
# for ans in answer:
#     if(ans[1]>best_score):
#         best_ans=ans[0].page_content
#         best_score=ans[1]
best_ans

'\u200bAll document loaders'

In [81]:
# generating output by using llm
from langchain_ollama import ChatOllama
llm=ChatOllama(model="gemma3:1b",temperature=0)

In [82]:
from langchain_core.prompts import ChatPromptTemplate
template='''
    Answer the question based ONLY on the following context:
    {context}
    Question: {question}
'''
prompt=ChatPromptTemplate.from_template(template)

In [83]:
context=best_ans
chain=prompt|llm

In [84]:
final_ans=chain.invoke({"context":context,"question":question})

In [85]:
print(final_ans.content)

The context states that “All document loaders” are needed to “handle documents of various formats.”
